# Brain Tumor Detection with YOLO — Your Object Detection Notebook

Welcome! Classification tells you *what* is in an image; **detection** goes further and tells you *where* it is, by drawing a box around it. In this notebook you'll train a real detector (**YOLO26**) to find brain tumors in MRI slices, and evaluate it the way detection is actually reported: with **Intersection over Union (IoU)** — how well a predicted box overlaps the true one — and an **FROC curve** — sensitivity vs false positives per image.

**What you'll do, step by step:**
1. Install the tools and check the environment (GPU or CPU)
2. Train YOLO26 on a brain-tumor MRI dataset
3. Play with the confidence threshold and watch **IoU**, sensitivity, and false positives update live on real images
4. See how the whole class's threshold choices trace out an FROC curve together

You don't need to know how to code to follow along — just run each cell in order and read the notes above it.

## Before you start — a quick checklist

- **Google account**: Colab needs a Google account to run code. Check the top-right corner of this page — if you're not signed in, sign in now (or create a free account at [accounts.google.com/signup](https://accounts.google.com/signup) before the session).
- **Your own copy**: if you opened this notebook from a shared link, you already have your own working copy — nothing you do here affects your classmates. If you'd like to keep this notebook after today, use **File → Save a copy in Drive**.
- **Turn on the GPU** (recommended — training is much faster): go to **Runtime → Change runtime type → Hardware accelerator → GPU**, then click Save.
- **Running cells**: click the ▶ button on the left of a cell, or select it and press **Shift+Enter**. Run the cells **in order, top to bottom** — later cells depend on earlier ones.
- **If something breaks**: don't panic. Go to **Runtime → Restart session**, then run all cells again from the top (**Runtime → Run all**).

```text
    _____   ________________    __    __ 
   /  _/ | / / ___/_  __/   |  / /   / / 
   / //  |/ /\__ \ / / / /| | / /   / /  
 _/ // /|  /___/ // / / ___ |/ /___/ /___
/___/_/ |_//____//_/ /_/  |_/_____/_____/
                                         
```

## Step 1: Install the tools

Now we install **Ultralytics** (the library behind YOLO) and check what hardware this session has — CPU or GPU. Just run the cell and read the printed report; there's nothing to change here.

In [ ]:
#@title ⚙ Install Ultralytics and run environment checks
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

```text
  __________  ___    _____   __
 /_  __/ __ \/   |  /  _/ | / /
  / / / /_/ / /| |  / //  |/ / 
 / / / _, _/ ___ |_/ // /|  /  
/_/ /_/ |_/_/  |_/___/_/ |_/   
                               
```

## Step 2: Train YOLO26 on brain-tumor MRI slices

This downloads a small MRI dataset (each image labeled with a box around any tumor) and trains YOLO26 to detect tumors on its own. **epochs** controls how many times the model sees the whole training set — more epochs usually means a better-trained model, but takes longer. Training will print progress for each epoch; when it finishes, `model` holds your trained detector, ready for Step 3.

In [ ]:
#@title 🚆 Train YOLO26 on the brain-tumor dataset
epochs = 25 #@param {type:"slider", min:5, max:60, step:5}
imgsz  = 640 #@param [320, 512, 640] {type:"raw"}

from ultralytics import YOLO
model = YOLO("yolo26n.pt")                       # COCO-pretrained nano detector
results = model.train(data="brain-tumor.yaml", epochs=epochs, imgsz=imgsz)

```text
    _______  __ ____  __________  ______  __________   ________
   / ____/ |/ // __ \/ ____/ __ \/  _/  |/  / ____/ | / /_  __/
  / __/  |   // /_/ / __/ / /_/ // // /|_/ / __/ /  |/ / / /   
 / /___ /   |/ ____/ /___/ _, _// // /  / / /___/ /|  / / /    
/_____//_/|_/_/   /_____/_/ |_/___/_/  /_/_____/_/ |_/ /_/     
                                                               
```

## Step 3: See detection metrics on real images

This is the heart of the lab. One slider, **conf (confidence)**, controls how the model's raw predictions become "detections": for every box it draws, YOLO also outputs a number between 0 and 1 — how sure *it* is that box really contains a tumor. This slider throws away any box below that confidence — raise it and only the model's most confident guesses survive; lower it and even shaky, uncertain guesses get a chance to count.

A kept box still has to be in the *right place* to be useful, which is a separate question from how confident the model was. That's measured by **Intersection over Union (IoU)** — *the* standard metric for object detection: the overlap between a predicted box and the true box, as a fraction between 0 (no overlap) and 1 (perfect match). Every box in the collage below is labelled with its own IoU, and this lab uses a **fixed threshold of 0.5** (the standard value for detection tasks) to decide which boxes count as correct: blue boxes (true positives) cleared it, red boxes (false positives) didn't — some barely missed it, some had none at all. The panel on the left also reports the **mean IoU** across all matched boxes, alongside sensitivity and false positives per image (which together make up the FROC curve, since a plain ROC curve doesn't work for detection — there's no fixed number of "true negatives" to compare against).

**Missed tumors matter too.** A ground-truth box (green) that no prediction ever reaches is a **false negative** — a real tumor the model missed entirely — shown with an **orange** outline instead of green, and counted in the title as "FN". Clinically, this is usually the outcome you care about most: a confident false alarm is annoying, a missed tumor can be dangerous. Pick **"false negatives"** in the **show** dropdown to jump straight to the cases your current settings are missing.

Try moving **conf** up and down and watch boxes flip colour as the confidence requirement changes — that's the sensitivity/false-positive/missed-tumor trade-off made concrete. When you're happy with a setting, click **Log this point** to add it to the class's shared results.

In [ ]:
#@title ▶ Interactive collage + FROC logger (sliders beside the images)
weights_path = "runs/detect/train/weights/best.pt"   # used only if no trained model is in memory

import os, time, datetime, math, random, numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as patches
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox
import ipywidgets as W
from IPython.display import display, clear_output
from ultralytics import YOLO

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

if "model" not in globals():
    if not os.path.exists(weights_path):
        raise FileNotFoundError("No `model` in memory and no weights at '" + weights_path +
                                "'. Run TRAIN first, or upload a best.pt and set weights_path.")
    model = YOLO(weights_path)

def find_val():
    cands = []
    try:
        from ultralytics.utils import SETTINGS
        cands.append(Path(SETTINGS.get("datasets_dir", ".")) / "brain-tumor")
    except Exception:
        pass
    cands += [Path("datasets/brain-tumor"), Path("/content/datasets/brain-tumor"),
              Path.home() / "datasets/brain-tumor"]
    for c in cands:
        if (c / "images/val").exists():
            return c / "images/val", c / "labels/val"
    raise FileNotFoundError("brain-tumor val split not found - run TRAIN first.")
val_img, val_lbl = find_val()
imgs = sorted(p for p in val_img.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
POS = 1

def gt_pos_boxes(p, w, h):
    boxes = []
    lp = val_lbl / (p.stem + ".txt")
    if lp.exists():
        for line in lp.read_text().splitlines():
            t = line.split()
            if len(t) >= 5 and int(float(t[0])) == POS:
                cx, cy, bw, bh = map(float, t[1:5])
                boxes.append([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h])
    return np.array(boxes, dtype=float) if boxes else np.zeros((0, 4))

def iou_xyxy(a, b):
    x1 = np.maximum(a[0], b[:, 0]); y1 = np.maximum(a[1], b[:, 1])
    x2 = np.minimum(a[2], b[:, 2]); y2 = np.minimum(a[3], b[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[2]-a[0]) * (a[3]-a[1]); ab = (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])
    return inter / (aa + ab - inter + 1e-9)

# ---- score the whole val set ONCE at low conf, then cache ----
print("Scoring the validation set once (a few seconds)...")
cache = []
preds = model.predict([str(p) for p in imgs], conf=0.001, verbose=False, stream=True)
for p, r in zip(imgs, preds):
    h, w = r.orig_shape
    gts = gt_pos_boxes(p, w, h)
    if r.boxes is not None and len(r.boxes):
        cls = r.boxes.cls.cpu().numpy(); cf = r.boxes.conf.cpu().numpy(); xy = r.boxes.xyxy.cpu().numpy()
        m = cls == POS; pboxes = xy[m]; pconf = cf[m]
    else:
        pboxes = np.zeros((0, 4)); pconf = np.zeros((0,))
    cache.append({"path": str(p), "gts": gts, "pboxes": pboxes, "pconf": pconf})
print("Ready - move the sliders.")

pos_idx = [i for i, c in enumerate(cache) if len(c["gts"]) > 0]
neg_idx = [i for i, c in enumerate(cache) if len(c["gts"]) == 0]
random.Random(0).shuffle(pos_idx); random.Random(1).shuffle(neg_idx)

def match(gts, pboxes, pconf, conf, iou_thr):
    """For every kept prediction box, find its IoU with the best-matching ground-truth box.
    A box counts as a true positive only if that IoU clears iou_thr AND that ground-truth box
    isn't already claimed by a higher-confidence box. iou_vals is returned for EVERY box (TP or
    FP), so you can see how close a false positive actually came to counting. `matched` marks,
    for each ground-truth box, whether some prediction found it (False = missed = a false
    negative - a real tumor the model didn't flag at all)."""
    keep = pconf >= conf
    pb = pboxes[keep]; pc = pconf[keep]
    pb = pb[np.argsort(-pc)]
    matched = np.zeros(len(gts), dtype=bool); flags = []; iou_vals = []; tp = fp = 0
    for box in pb:
        is_tp = False; iou_val = 0.0
        if len(gts):
            ious = iou_xyxy(box, gts); j = int(np.argmax(ious))
            iou_val = float(ious[j])
            if ious[j] >= iou_thr and not matched[j]:
                matched[j] = True; is_tp = True
        flags.append(is_tp); iou_vals.append(iou_val)
        tp += int(is_tp); fp += int(not is_tp)
    return tp, fp, pb, flags, iou_vals, matched

def froc_point(conf, iou_thr):
    TP = FP = GT = 0
    all_tp_ious = []
    for c in cache:
        tp, fp, _, flags, iou_vals, _ = match(c["gts"], c["pboxes"], c["pconf"], conf, iou_thr)
        TP += tp; FP += fp; GT += len(c["gts"])
        all_tp_ious.extend(iv for iv, f in zip(iou_vals, flags) if f)
    mean_iou = float(np.mean(all_tp_ious)) if all_tp_ious else float("nan")
    return (TP / GT if GT else 0.0), FP / len(cache), TP, FP, GT, mean_iou

def pick_indices(focus, conf, iou_thr, n):
    if focus == "tumor present":
        return sorted(pos_idx[:n])
    if focus == "tumor absent":
        return sorted(neg_idx[:n]) if neg_idx else sorted(pos_idx[:n])
    if focus == "false negatives":
        # rank images by number of MISSED tumours (ground-truth boxes with no matching prediction)
        fnc = []
        for i, c in enumerate(cache):
            _, _, _, _, _, matched = match(c["gts"], c["pboxes"], c["pconf"], conf, iou_thr)
            n_fn = int((~matched).sum()) if len(matched) else 0
            if n_fn > 0:
                fnc.append((n_fn, i))
        fnc.sort(reverse=True)
        fn_idx = [i for _, i in fnc]
        return sorted(fn_idx[:n]) if fn_idx else sorted(pos_idx[:n])
    # rank images by number of false positives at the current thresholds
    fpc = []
    for i, c in enumerate(cache):
        _, fp, _, _, _, _ = match(c["gts"], c["pboxes"], c["pconf"], conf, iou_thr)
        if fp > 0:
            fpc.append((fp, i))
    fpc.sort(reverse=True)
    fp_idx = [i for _, i in fpc]
    if focus == "false positives":
        return sorted(fp_idx[:n]) if fp_idx else sorted(pos_idx[:n])
    # mixed: half tumour cases, half worst false-positive cases
    half = max(1, n // 2)
    a = pos_idx[:half]
    b = [i for i in fp_idx if i not in a][:n - len(a)]
    return sorted(a + b) if (a or b) else sorted(pos_idx[:n])

# ---- widgets: controls on the left, collage on the right ----
name_w = W.Text(value="", description="name")
conf_s = W.FloatSlider(value=0.25, min=0.05, max=0.90, step=0.05, description="conf", continuous_update=False)
IOU_THRESHOLD = 0.5  # fixed - standard detection threshold, not a tunable slider (avoids being mistaken for a training hyperparameter)
n_dd   = W.Dropdown(options=[4, 6, 9], value=6, description="images")
focus_dd = W.Dropdown(options=["tumor present", "false positives", "false negatives", "tumor absent", "mixed"], value="tumor present", description="show")
log_btn = W.Button(description="Log this point", button_style="success", icon="check")
status = W.HTML()
out = W.Output()

SHEET_URL = "https://docs.google.com/spreadsheets/d/1_-kntyt1fDtnypsKNcVb9zEVrtoevlwu3sPigNH5-hs/edit?usp=sharing"
HEADER = ["timestamp", "name", "conf", "iou_thr", "sensitivity", "fp_per_image", "mean_iou", "n_tp", "n_fp"]

def colored_title(ax, gtn, tpn, fnn, fpn, fontsize=8, y=1.02):
    # multicolour title matching the box colours: GT green, TP blue, FN orange, FP red
    parts = [("GT %d" % gtn, "lime"), ("  |  ", "0.5"),
             ("TP %d" % tpn, "deepskyblue"), ("  |  ", "0.5"),
             ("FN %d" % fnn, "orange"), ("  |  ", "0.5"),
             ("FP %d" % fpn, "red")]
    tbxs = [TextArea(t, textprops=dict(color=col, fontsize=fontsize, weight="bold")) for t, col in parts]
    ab = AnnotationBbox(HPacker(children=tbxs, align="center", pad=0, sep=0),
                        (0.5, y), xycoords="axes fraction", frameon=False, box_alignment=(0.5, 0))
    ax.add_artist(ab)

def render(*_):
    sens, fppi, TP, FP, GT, mean_iou = froc_point(conf_s.value, IOU_THRESHOLD)
    iou_txt = ("%.2f" % mean_iou) if mean_iou == mean_iou else "n/a"   # NaN check
    status.value = ("<b>conf=%.2f</b><br>sensitivity = <b>%.3f</b>"
                    "<br>FP / image = <b>%.3f</b><br>mean IoU (matched boxes) = <b>%s</b>"
                    "<br><span style='color:gray'>TP=%d/%d&nbsp;FP=%d</span>"
                    % (conf_s.value, sens, fppi, iou_txt, TP, GT, FP))
    idxs = pick_indices(focus_dd.value, conf_s.value, IOU_THRESHOLD, n_dd.value)
    with out:
        clear_output(wait=True)
        ncol = 3; nrow = math.ceil(len(idxs) / ncol)
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.3*ncol, 3.3*nrow))
        axes = np.array(axes).reshape(-1)
        for ax, i in zip(axes, idxs):
            c = cache[i]
            ax.imshow(Image.open(c["path"]).convert("RGB")); ax.axis("off")
            tp, fp, pb, flags, iou_vals, matched = match(c["gts"], c["pboxes"], c["pconf"], conf_s.value, IOU_THRESHOLD)
            for g, was_found in zip(c["gts"], matched):
                gt_col = "lime" if was_found else "orange"   # orange = missed tumour (false negative)
                ax.add_patch(patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1],
                                               fill=False, edgecolor=gt_col, lw=2))
            for box, tpf, iouv in zip(pb, flags, iou_vals):
                col = "deepskyblue" if tpf else "red"
                ax.add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                               fill=False, edgecolor=col, lw=2))
                ax.text(box[0], max(box[1] - 4, 0), "IoU %.2f" % iouv, color=col,
                       fontsize=6, va="bottom", ha="left")
            fn = int((~matched).sum()) if len(matched) else 0
            colored_title(ax, len(c["gts"]), tp, fn, fp)
        for ax in axes[len(idxs):]:
            ax.axis("off")
        plt.tight_layout(); plt.show()

def on_log(_):
    try:
        from google.colab import auth; auth.authenticate_user()
        import gspread
        from google.auth import default
        creds, _ = default(); gc = gspread.authorize(creds)
        ws = gc.open_by_url(SHEET_URL).sheet1
        values = ws.get_all_values()
        if not values:
            ws.append_row(HEADER)
        elif values[0] != HEADER:
            status.value += "<br><span style='color:red'>Sheet header mismatch - clear the sheet and retry.</span>"
            return
        sens, fppi, TP, FP, GT, mean_iou = froc_point(conf_s.value, IOU_THRESHOLD)
        mean_iou_val = round(mean_iou, 4) if mean_iou == mean_iou else ""   # NaN -> blank cell
        row = [datetime.datetime.now().isoformat(timespec="seconds"), name_w.value or "anon",
               conf_s.value, IOU_THRESHOLD, round(sens, 4), round(fppi, 4), mean_iou_val, TP, FP]
        ws.append_row(row, value_input_option="USER_ENTERED")
        status.value += "<br><span style='color:green'>Logged ✓ (conf=%.2f)</span>" % (conf_s.value,)
    except Exception as e:
        status.value += "<br><span style='color:red'>Log failed: %s</span>" % e

for wdg in (conf_s, n_dd, focus_dd):
    wdg.observe(render, "value")
log_btn.on_click(on_log)

controls = W.VBox([name_w, conf_s, n_dd, focus_dd, log_btn, status])
display(W.HBox([controls, out]))
render()

```text
    ____  ___   _____ __  ______  ____  ___    ____  ____ 
   / __ \/   | / ___// / / / __ )/ __ \/   |  / __ \/ __ \
  / / / / /| | \__ \/ /_/ / __  / / / / /| | / /_/ / / / /
 / /_/ / ___ |___/ / __  / /_/ / /_/ / ___ |/ _, _/ /_/ / 
/_____/_/  |_/____/_/ /_/_____/\____/_/  |_/_/ |_/_____/  
                                                          
```

## Step 4 (optional): See how the whole class did

This part is optional and doesn't affect your own results above. If your instructor has set up a shared Google Sheet, every logged point — sensitivity, false positives per image, and mean IoU — is added to it. Running the cell below plots everyone's points together with a single fitted curve, so you can see the class's collective FROC curve emerge from individual threshold choices.

If this cell shows an error (e.g. about the sheet), don't worry — it doesn't mean your detector failed. Your results from Step 3 are already valid on their own.

In [ ]:
#@title 📊 Class FROC: point cloud + single class fit
from google.colab import auth; auth.authenticate_user()
import gspread, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.auth import default
creds, _ = default(); gc = gspread.authorize(creds)
SHEET_URL = "https://docs.google.com/spreadsheets/d/1_-kntyt1fDtnypsKNcVb9zEVrtoevlwu3sPigNH5-hs/edit?usp=sharing"
ws = gc.open_by_url(SHEET_URL).sheet1

try:
    records = ws.get_all_records(value_render_option="UNFORMATTED_VALUE")
except TypeError:
    records = ws.get_all_records()
df = pd.DataFrame(records)

need = {"conf", "sensitivity", "fp_per_image"}
if df.empty:
    print("No data yet - run the EXPERIMENT cell first.")
elif not need.issubset(df.columns):
    print("Unexpected columns in the sheet:", list(df.columns))
    print("This sheet holds data from an earlier version. Clear it (select all > Delete) and re-log.")
else:
    cols = ["conf", "sensitivity", "fp_per_image"] + (["iou_thr"] if "iou_thr" in df.columns else [])
    for c in cols:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df = df.dropna(subset=["sensitivity", "fp_per_image"])
    if df.empty:
        print("Rows found but the numbers could not be parsed - check the sheet values.")
    else:
        x = df["fp_per_image"].to_numpy(); y = df["sensitivity"].to_numpy()
        plt.figure(figsize=(7, 5))

        # point cloud (no connecting lines); colour by confidence threshold if available
        # (iou_thr is now fixed for everyone, so it is no longer a useful colour dimension)
        if "conf" in df.columns and df["conf"].notna().any():
            sc = plt.scatter(x, y, c=df["conf"], cmap="viridis", s=45, alpha=.8,
                             edgecolor="k", linewidth=.3)
            plt.colorbar(sc, label="confidence threshold")
        else:
            plt.scatter(x, y, s=45, alpha=.8, edgecolor="k", linewidth=.3)

        # single saturating fit across ALL class points
        def sat(xx, a, k): return a * (1.0 - np.exp(-k * xx))
        if len(df) >= 3 and np.ptp(x) > 0:
            try:
                from scipy.optimize import curve_fit
                p0 = [min(1.0, max(float(y.max()), 0.5)), 1.0]
                popt, _ = curve_fit(sat, x, y, p0=p0, bounds=([0, 0], [1.0, np.inf]), maxfev=10000)
                xs = np.linspace(0, float(x.max()) * 1.05, 200)
                yhat = sat(x, *popt)
                ss_res = float(np.sum((y - yhat) ** 2)); ss_tot = float(np.sum((y - y.mean()) ** 2))
                r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
                plt.plot(xs, sat(xs, *popt), "-", color="crimson", lw=2,
                         label="class fit: a=%.2f, k=%.2f (R²=%.2f)" % (popt[0], popt[1], r2))
                plt.legend(loc="lower right")
            except Exception as e:
                print("Fit skipped:", e)
        else:
            print("Need at least 3 points spread over different FP/image values to fit a curve.")

        plt.xlabel("False positives per image"); plt.ylabel("Sensitivity (per lesion)")
        plt.ylim(0, 1.02); plt.xlim(left=0)
        plt.title("Class FROC - %d operating points" % len(df))
        plt.grid(alpha=.3); plt.show()